# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

We'll print record set summaries, including their `@id`, label, and available fields or columns.

In [ ]:
# List available record sets, with their `@id` and fields/columns
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this dataset. Please check Croissant schema definitions.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs.id}")
        if hasattr(rs, 'label') and rs.label:
            print(f"  Label: {rs.label}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for field in rs.fields:
                print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', '')}")
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                print(f"    - Column @id: {col.id}, name: {getattr(col, 'name', '')}")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

All entities, including record sets, fields, and columns, are referenced by their `@id` fields as per best practice.

In [ ]:
# Find all record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
        print()
    except Exception as e:
        print(f"Could not load records for record set @id {record_set_id}: {e}")

# If at least one record set was loaded, display its columns and preview
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"Columns in first record set (@id: {first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
This section demonstrates common data processing steps on one record set, using fields referenced by their `@id`.

You'll learn how to filter records based on a numeric field, normalize that field, and group data by a categorical field, all identified by their `@id`.

In [ ]:
# If any data was loaded, perform EDA on the first record set found
if dataframes:
    # Choose the first available record set
    record_set_id = first_rs_id
    df = dataframes[record_set_id].copy()
    print(f"Running EDA on record set @id: {record_set_id}")

    # Attempt to auto-detect a numeric field (by dtype or by common names)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        # If not found, prompt user
        print("No numeric fields detected. Please verify field mappings.")
    else:
        print(f"Analyzing numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() > 0 else 0
        # Filter records where the numeric field is above the mean
        filtered_df = df[df[numeric_field_id] > threshold].copy() if threshold else df.copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, col_norm]].head())

        # Attempt to auto-detect a categorical/group field (non-numeric)
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (means):")
            print(grouped_df.head())
else:
    print("No dataframes found. Please check previous steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll create a histogram for the numeric field and, if possible, a bar plot by group.

All fields/columns are referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[first_rs_id]

    # Reuse previously detected field IDs if available
    try:
        numeric_field_id
    except NameError:
        numeric_field_id = None
    try:
        group_field_id
    except NameError:
        group_field_id = None

    # Histogram of numeric field
    if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_field_id], kde=True, bins=20, color='teal')
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    # Barplot of group means if group/categorical field available
    if group_field_id and numeric_field_id and group_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_means.index, y=group_means.values, palette="viridis")
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No loaded dataframes for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and inspect Croissant-structured metadata and record sets using the `mlcroissant` API,
- Reference all dataset objects by their `@id`,
- Load tabular data and automatically explore its fields,
- Apply basic data processing and grouping by identifiers,
- Visualize distributions using field `@id`s for full traceability.

The FAIR² dataset provides regression outputs and survey-based observations on adoption predictors in rangeland management. You can build on this workflow to perform further domain-specific analyses!